<a href="https://colab.research.google.com/github/01460421/01460421.github.io/blob/main/composer_style/pjsk_composer_style.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PJSK 作曲家風格量化

從遊戲音源抽出節奏、音色、和聲、情緒等特徵，再整理成每位作曲家的風格輪廓。

- **快速模式**（預設）：直接使用 repo 裡已經算好的特徵（`composer_style/data/`），幾秒就能出報告。
- **完整模式**：重新下載全部音源並抽取特徵（CPU 約 1 小時），適合新歌上線後更新資料。

In [ ]:
#@title 1. 安裝套件與下載程式
QUICK = True  #@param {type:"boolean"}
!pip -q install librosa essentia-tensorflow
!apt-get -qq install -y fonts-noto-cjk > /dev/null
!rm -rf repo && git clone -q --depth 1 https://github.com/01460421/01460421.github.io.git repo
%cd repo/composer_style

In [ ]:
#@title 2. 下載 Essentia 模型（約 25 MB）
!mkdir -p models
B = "https://essentia.upf.edu/models"
files = ["feature-extractors/discogs-effnet/discogs-effnet-bs64-1.pb",
         "classification-heads/genre_discogs400/genre_discogs400-discogs-effnet-1.pb",
         "classification-heads/genre_discogs400/genre_discogs400-discogs-effnet-1.json"]
for h in ["mood_happy", "mood_sad", "mood_aggressive", "mood_relaxed",
          "danceability", "mood_acoustic", "mood_electronic"]:
    files.append(f"classification-heads/{h}/{h}-discogs-effnet-1.pb")
for f in files:
    !wget -q -nc -P models {B}/{f}
!ls models

In [ ]:
#@title 3. 抽取特徵（快速模式會跳過）
# 完整模式建議把 OUT 設在 Google Drive，斷線後重跑會從 cache 接續
OUT = "data"  #@param {type:"string"}
if not QUICK:
    !python extract.py --models models --out {OUT} --workers 2
else:
    print("快速模式：使用 repo 內已計算的特徵")

In [ ]:
#@title 4. 產生作曲家報告
MIN_SONGS = 3  #@param {type:"integer"}
!python analyze.py --data {OUT} --models models --report report --min-songs {MIN_SONGS}
from IPython.display import Markdown, Image, display
display(Image("report/fig_style_map.png"))
display(Image("report/fig_heatmap.png"))

In [ ]:
#@title 5. 查詢單一作曲家
COMPOSER = "DECO*27"  #@param {type:"string"}
import re
text = open("report/composer_profiles.md").read()
m = re.search(rf"### {re.escape(COMPOSER)}（.*?(?=\n### |\Z)", text, re.S)
display(Markdown(m.group(0) if m else f"找不到「{COMPOSER}」（歌曲數可能少於 {MIN_SONGS} 首）"))

In [ ]:
#@title 6. 兩位作曲家雷達圖比較
A = "DECO*27"  #@param {type:"string"}
B_ = "かいりきベア"  #@param {type:"string"}
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager
import analyze
for name in ["Noto Sans CJK JP", "IPAGothic"]:
    if any(f.name == name for f in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = name; break
prof = pd.read_csv("report/composer_profiles.csv").set_index("composer")
keys = ["bpm", "onset_rate", "percussive_ratio", "brightness", "flatness", "bass_ratio",
        "dynamic_range_db", "harmonic_entropy", "chord_change", "is_minor",
        "happy", "sad", "aggressive", "danceability", "electronic", "acoustic"]
labels = [analyze.TRAITS[k][0] for k in keys]
ang = np.linspace(0, 2 * np.pi, len(keys), endpoint=False).tolist(); ang += ang[:1]
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
for who in [A, B_]:
    v = prof.loc[who, keys].clip(-2, 2).tolist(); v += v[:1]
    ax.plot(ang, v, label=f"{who}（{int(prof.loc[who, 'songs'])} 首）"); ax.fill(ang, v, alpha=.15)
ax.set_xticks(ang[:-1], labels, fontsize=9); ax.set_ylim(-2, 2)
ax.set_title("z 分數（0 = PJSK 平均）"); ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.show()

In [ ]:
#@title 7. 找聲音最像某首歌的其他歌
TITLE = "ロキ"  #@param {type:"string"}
df = pd.read_csv(f"{OUT}/features.csv")
emb = np.load(f"{OUT}/embed.npy")
E = emb / np.linalg.norm(emb, axis=1, keepdims=True); E -= E.mean(0)
E /= np.linalg.norm(E, axis=1, keepdims=True)
hit = df.index[df.title == TITLE]
if len(hit) == 0:
    print("找不到這首歌，可以用部分標題搜尋：", df[df.title.str.contains(TITLE[:2], regex=False)].title.tolist()[:10])
else:
    s = E @ E[hit[0]]
    top = np.argsort(-s)[1:11]
    display(pd.DataFrame({"歌曲": df.title.iloc[top].values, "作曲": df.composer.iloc[top].values,
                          "相似度": s[top].round(3)}))